# Source selection · historical investigation

**Optional background, outside the main run path.** These five cells preserve the initial alternative-provider exploration. They explain the coverage checks that preceded selecting IMPECT Bundesliga 2023/24. Re-running uses the provider’s current catalogue, so source listings can change.

Run cells from top to bottom in a **fresh kernel**. The setup cell declares inputs explicitly. Code comments and explanations are in English. The original source cell numbers are recorded in notebook metadata and in `docs/cell_mapping.csv`.

## Setup

Locate the repository and import shared utilities. This does not change the database.

In [ ]:
from pathlib import Path
import sys
import json
from urllib.request import urlopen
import numpy as np
import pandas as pd
from IPython.display import display

# Find the repository from any notebook subfolder.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "scouting" / "config.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook inside the extracted repository.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from scouting.config import PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, REPORTS_DIR, IMPECT_BASE_URL, ITERATION_ID
from scouting.data_access import load_or_download_json, load_metadata_context, metadata
from scouting.checkpoints import save_context, load_context, BASE_VARIABLES, ADVANCED_VARIABLES
print("Project:", PROJECT_ROOT)



## Read an alternative provider catalogue

Load competition/season metadata from the earlier source investigation. This is historical exploration, not an input to the final IMPECT analysis.

In [ ]:
import json
from urllib.request import urlopen

import pandas as pd

BASE_URL = "https://raw.githubusercontent.com/hudl/open-data/master/data"

with urlopen(f"{BASE_URL}/competitions.json", timeout=30) as response:
    competitions_data = json.load(response)

competitions = pd.DataFrame(competitions_data)

print("Competition-season entries:", len(competitions))
competitions.head()

## Filter the catalogue

Keep senior men’s domestic competitions to match the scouting scope.

In [ ]:
# Keep men's senior domestic competitions.
mask = (
    competitions["competition_gender"].eq("male")
    & competitions["competition_youth"].eq(False)
    & competitions["competition_international"].eq(False)
)

columns = [
    "competition_id",
    "season_id",
    "country_name",
    "competition_name",
    "season_name",
]

available_competitions = (
    competitions.loc[mask, columns]
    .sort_values(["competition_name", "season_name"])
    .reset_index(drop=True)
)

display(available_competitions)

## Inspect available matches

Read the proposed season’s match list. A season label alone does not establish full coverage.

In [ ]:
competition_id = 9
season_id = 281

matches_url = (
    f"{BASE_URL}/matches/{competition_id}/{season_id}.json"
)

with urlopen(matches_url, timeout=30) as response:
    matches_data = json.load(response)

matches = pd.json_normalize(matches_data)

print("Match records:", len(matches))
print("Unique matches:", matches["match_id"].nunique())

display(
    matches[
        [
            "match_id",
            "match_date",
            "home_team.home_team_name",
            "away_team.away_team_name",
            "home_score",
            "away_score",
        ]
    ].head()
)

## Measure team coverage

Count home and away appearances per team to detect partial season coverage.

In [ ]:
home_appearances = matches[
    ["match_id", "home_team.home_team_name"]
].rename(columns={"home_team.home_team_name": "team"})

away_appearances = matches[
    ["match_id", "away_team.away_team_name"]
].rename(columns={"away_team.away_team_name": "team"})

team_appearances = pd.concat(
    [home_appearances, away_appearances],
    ignore_index=True,
)

team_coverage = (
    team_appearances.groupby("team")["match_id"]
    .nunique()
    .reset_index(name="available_matches")
    .sort_values(["available_matches", "team"])
    .reset_index(drop=True)
)

print("Teams represented:", len(team_coverage))
display(team_coverage)

## Compare alternative seasons

Compare four source options. The older Premier League season was complete, but we chose a more recent full Bundesliga season from IMPECT.

In [ ]:
candidates = [
    ("Premier League", "2015/2016", 2, 27),
    ("La Liga", "2020/2021", 11, 90),
    ("Ligue 1", "2022/2023", 7, 235),
    ("Major League Soccer", "2023", 44, 107),
]

coverage_results = []

for competition, season, comp_id, seas_id in candidates:
    url = f"{BASE_URL}/matches/{comp_id}/{seas_id}.json"

    with urlopen(url, timeout=30) as response:
        data = json.load(response)

    match_table = pd.json_normalize(data)
    match_table = match_table.drop_duplicates(subset="match_id")

    # Each match contributes one appearance for each team.
    teams = pd.concat(
        [
            match_table["home_team.home_team_name"],
            match_table["away_team.away_team_name"],
        ],
        ignore_index=True,
    )

    counts = teams.value_counts()

    coverage_results.append({
        "competition": competition,
        "season": season,
        "matches": len(match_table),
        "teams": len(counts),
        "min_matches_per_team": counts.min(),
        "max_matches_per_team": counts.max(),
    })

coverage_comparison = pd.DataFrame(coverage_results)

display(coverage_comparison)